# Caracal Bench s01 · TPU one-shot

Roda em TPU. Mede checkpoint `caracal-base-3b-s01` vs Qwen base em:
- probe set (51 prompts) - ppl + CWE hit
- MMLU computer_security (~100 MCQ) - acc
- HumanEval (164 problems) - pass@1

**Antes de Run All:**
1. Settings -> Accelerator -> TPU VM v3-8 (ou v5e-8)
2. Settings -> Internet -> ON
3. Settings -> Persistence -> Variables and Files
4. Edita CHECKPOINT_DATASET + OUTPUT_DATASET abaixo se preciso

Tempo esperado: ~1.5h (compile XLA ~5min + ~45min gen adapter + ~45min gen base).

In [ ]:
CHECKPOINT_DATASET = "pedroafonso2/caracal-base-3b-s01"
OUTPUT_DATASET = "pedroafonso2/caracal-bench-s01"
# TPU XLA recompila a cada shape no .generate() = inviavel.
# Probe (51 prompts) + HumanEval (164) usam generate -> rodar em GPU T4.
# MMLU = single forward pass shape-fixo -> roda OK em TPU (~7min/100 questoes).
EXTRA_FLAGS = ["--skip-probe", "--skip-humaneval"]
print(f"checkpoint={CHECKPOINT_DATASET} -> TPU bench (MMLU only)")

In [ ]:
!pip install -q 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' kaggle

In [ ]:
import os
import subprocess

if not os.path.exists("/kaggle/working/caracal-1"):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "-b",
            "dev",
            "https://github.com/iterate-labs-ai/caracal-1.git",
            "/kaggle/working/caracal-1",
        ],
        check=True,
    )
os.chdir("/kaggle/working/caracal-1")
rev = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"cloned, HEAD={rev}", flush=True)

In [ ]:
import subprocess

ckpt_dir = "/kaggle/working/ckpt-s01"
subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        CHECKPOINT_DATASET,
        "-p",
        ckpt_dir,
        "--unzip",
        "--force",
    ],
    check=True,
)
subprocess.run(["ls", "-la", ckpt_dir], check=True)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",
    "eval/run_bench_all.py",
    "--adapter",
    ckpt_dir,
    "--out-dir",
    "/kaggle/working/bench-s01",
]
if SKIP_PROBE:
    cmd.append("--skip-probe")
if SKIP_HUMANEVAL:
    cmd.append("--skip-humaneval")
print("Running adapter bench:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",
    "eval/run_bench_all.py",
    "--adapter",
    ckpt_dir,
    "--out-dir",
    "/kaggle/working/bench-s01",
] + EXTRA_FLAGS
print("Running adapter bench:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",
    "eval/run_bench_all.py",
    "--out-dir",
    "/kaggle/working/bench-base",
] + EXTRA_FLAGS
print("Running base bench:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
import json
import subprocess
from pathlib import Path

metadata = {
    "title": "Caracal Bench s01 vs Base",
    "id": OUTPUT_DATASET,
    "licenses": [{"name": "Apache-2.0"}],
}
(pub_dir / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

cmd_create = ["kaggle", "datasets", "create", "-p", str(pub_dir), "--public"]
cmd_version = ["kaggle", "datasets", "version", "-p", str(pub_dir), "-m", "bench s01"]

r = subprocess.run(cmd_create, capture_output=True, text=True, check=False)
print(r.stdout, r.stderr)
if r.returncode != 0:
    subprocess.run(cmd_version, check=True)
print(f"Published -> {OUTPUT_DATASET}")

## O que ler depois

```bash
kaggle datasets download -d pedroafonso2/caracal-bench-s01 -p . --unzip
cat bench_s01_vs_base.json | jq .deltas
```

**Sucesso v0**: `humaneval.delta >= 0` (nao regrediu code gen) E `probe.cwe_hit_rate.delta > 0` (aprendeu CWE) E `mmlu_security.acc.delta > 0` (aprendeu conhecimento cyber).